In [ ]:
!uv pip install morphik

## VNS Policy

In [ ]:
from morphik import Morphik

# Connect to Morphik
db = Morphik(uri=None, timeout=10000, is_local=True)

In [ ]:
import json

json_paths = {
    "vns": "/Users/mac/Documents/PHUNGPX/knowledge_graph_searching/examples/data/vns/VNS.json",
}


samples = []
for category, json_path in json_paths.items():
    with open(json_path, "r", encoding="utf-8") as file:
        data = json.load(file)
        print(f"Loading {json_path} - {len(data)} items")
        for item in data:
            if item.get("text", None) is not None:
                samples.append(
                    {"text": item["text"]["raw_text"], "metadata": {"category": category}}
                )
            if item.get("image", None) is not None:
                samples.append(
                    {"text": item["image"]["image_caption"], "metadata": {"category": category}}
                )
            if item.get("table", None) is not None:
                samples.append(
                    {"text": item["table"]["content"], "metadata": {"category": category}}
                )

In [ ]:
len(samples), samples[0]

In [ ]:
# Ingest the documents
doc_ids = []
for i, sample in enumerate(samples):
    doc = db.ingest_text(sample['text'], metadata=sample['metadata'], use_colpali=False)
    doc_ids.append(doc.external_id)
    print(f"{i + 1} / {len(samples)} - Ingested document with ID: {doc.external_id}")

In [ ]:
doc_id_to_text = {}
for i, doc_id in enumerate(doc_ids):
    doc = db.get_document(doc_id)
    doc_id_to_text[doc_id] = samples[i]

In [ ]:
doc_ids

In [ ]:
# Create a knowledge graph from the documents
print("Creating knowledge graph...")
graph = db.create_graph(name="vns_policy_version_3", documents=doc_ids)
print(f"Created graph with name: {graph.name}")

In [ ]:
viz = db.get_graph_visualization("vns_policy")
print(len(viz["nodes"]), "nodes", len(viz["links"]), "edges")

In [ ]:
viz = db.get_graph_visualization("updated_vns_policy")
print(len(viz["nodes"]), "nodes", len(viz["links"]), "edges")

In [ ]:
viz = db.get_graph_visualization("vns_policy_version_3")
print(len(viz["nodes"]), "nodes", len(viz["links"]), "edges")

In [ ]:
import time
from morphik import Morphik

db = Morphik(uri=None, timeout=50, is_local=True)

In [ ]:
questions = [
    "In the first-day “Next schedule”, what happens between 11 : 30 – 12 : 00?",
    "List any four of VNS’s core values.",
    "How many on-top annual-leave days do employees get, and how many can be carried over?",
    "Quote two purposes for which VNS collects personal information.",
    "What does the Policy call the unique files (‘cookies’) stored on your device?",
    "Who's the CEO of VNS",
    "Who holds the Managing Director role at VNS?",
    "On the first-day schedule, what activity happens from 10 : 15 – 11 : 30?"
]

for question in questions:
    t1 = time.time()
    response = db.query(
        question,
        k=4,
        graph_name="vns_policy_version_3",
        hop_depth=2,
        use_colpali=False,
        include_paths=True,
        # stream_response=False,
    )
    t2 = time.time()
    print("------------------------------")
    print(f"Question {question}")
    print(f"Answer: {response.completion}")
    print(f"Response Time: {t2 - t1} seconds")
    # print(f"Context: {response.context}")
    print(f"Paths: {response.metadata['graph']['paths']}")
    print("------------------------------")
    print("\n")
    

In [ ]:
# Example of using a graph with path information
response_with_paths = db.query(
    "Who's the CEO of VNS",
    k=4,
    graph_name="updated_vns_policy",
    hop_depth=2,
    use_colpali=False,
    include_paths=True,
)

In [ ]:
print(response_with_paths)

In [ ]:
print(response_with_paths.metadata['graph']['name'])
print(response_with_paths.metadata['graph']['relevant_entities'])
print(response_with_paths.metadata['graph']['paths'])

In [ ]:
# Print the sources used for the completion
for source in response.sources:
    print(f"Document ID: {source.document_id}, Chunk: {source.chunk_number}, Score: {source.score}")

In [ ]:
questions = [
    "What is the document’s most recent update date and effective date?",
    "How does the policy define Personal Data?",
    "What must you confirm about any referee information you add to your profile?",
    "Give three purposes for which VNS may process Personal Data without prior notice or consent.",
    "What right do you have if your Personal Data is outdated or inaccurate?",
    "Name two ways VNS collects Personal Data.",
    "Provide three technical data points VNS may collect when you use its websites or apps.",
    "State three purposes for which your personal information may be used.",
    "What security measures does VNS mention for protecting Personal Data?",
    "Who may your Personal Data be disclosed to? Give three categories.",
    "Where may your information be processed or stored geographically?",
    "What is the contact email for requests about accessing or correcting Personal Data?",
    "May VNS use your data for direct marketing? Under what condition?",
    "How can you opt out of direct marketing after consenting?",
    "What kinds of information can data processing tools (cookies) collect for analytics?",
    "What happens if you block the site’s data processing tools (cookies)?",
    "Does the policy permit automated processing of personal data?",
    "Are external websites linked from VNS covered by this Policy?",
    "Can VNS change this Policy without prior notice, and when do changes take effect?",
    "What is VNS’s mission?",
    "What are the core values of VNS?",
    "When was VNS licensed and when did the first office open?",
    "What industries does VNS serve?",
    "Who are the top executives of VNS?",
    "What are some key benefits and rewards offered by VNS?",
    "What are the official working hours at VNS?",
    "What does the first-day onboarding schedule include?",
]

for i, question in enumerate(questions):
    t1 = time.time()
    response = db.query(
        question,
        k=10,
        graph_name="updated_vns_policy",
        hop_depth=2,
        use_colpali=False,
        include_paths=True,
        # use_reranking=True,
        # stream_response=False,
    )
    t2 = time.time()
    print("------------------------------")
    print(f"Question #{i + 1} / {len(questions)}: {question}")
    print(f"Answer: {response.completion}")
    print(f"Response Time: {t2 - t1} seconds")
    # print(f"Paths: {response.metadata['graph']['paths']}")
    print("------------------------------")
    print("\n")

# Pest and Disease

In [46]:
from morphik import Morphik

# Connect to Morphik
db = Morphik(uri=None, timeout=100, is_local=True)

In [13]:
from pathlib import Path

json_paths = list(Path("/Users/mac/Documents/PHUNGPX/knowledge_graph_searching/examples/data/pest_and_disease").glob("*.json"))

print(f"The number of json paths: {len(json_paths)}")

The number of json paths: 17


In [14]:
import json

category = "durian_pest_and_disease_v1"

samples = []
for json_path in json_paths:
    with open(json_path, "r", encoding="utf-8") as file:
        data = json.load(file)
        print(f"Loading {json_path} - {len(data)} items")
        for item in data:
            if item.get("text", None) is not None:
                samples.append(
                    {"text": item["text"]["text_translated"], "metadata": {"category": category}}
                )
            if item.get("image", None) is not None:
                samples.append(
                    {"text": item["image"]["image_caption"], "metadata": {"category": category}}
                )
            if item.get("table", None) is not None:
                samples.append(
                    {"text": item["table"]["content"], "metadata": {"category": category}}
                )

Loading /Users/mac/Documents/PHUNGPX/knowledge_graph_searching/examples/data/pest_and_disease/PL-DR-DP-ED-09-หนอนเจาะเมล็ดทุเรียน.pdf_index.json - 1 items
Loading /Users/mac/Documents/PHUNGPX/knowledge_graph_searching/examples/data/pest_and_disease/PL-DR-DP-ED-06-เพลี้ยหอย.pdf_index.json - 1 items
Loading /Users/mac/Documents/PHUNGPX/knowledge_graph_searching/examples/data/pest_and_disease/PL-DR-DP-ED-08-มอดเจาะลำต้น.pdf_index.json - 1 items
Loading /Users/mac/Documents/PHUNGPX/knowledge_graph_searching/examples/data/pest_and_disease/PL-DR-DP-THE-03-โรคกิ่งแห้งของทุเรียนที่เกิดจากเชื้อรา Lasiodiplodia theobromae และการควบคุมโดยใช้สารเคมี.pdf_index.json - 71 items
Loading /Users/mac/Documents/PHUNGPX/knowledge_graph_searching/examples/data/pest_and_disease/DR-DP-ED-02-โรคทุเรียนและการป้องกันกำจัด.pdf_index.json - 6 items
Loading /Users/mac/Documents/PHUNGPX/knowledge_graph_searching/examples/data/pest_and_disease/DR-DP-ED-03-โรคของทุเรียน.pdf_index.json - 3 items
Loading /Users/mac/Docu

In [15]:
len(samples), samples[0]

(618,
 {'text': 'Durian seed borer (Durian seed borer). Nature of Damage: The larva bores into the durian fruit and excretes dark brown frass that covers the entry hole. It feeds on the seed inside the fruit. When the larva is fully grown and ready to pupate, or when food is insufficient, it will bore a hole out of the husk and drop to the ground to pupate in the soil or find a new food source to complete its life cycle. Scientific Name: Mudaria luteileprosa Holloway. Other names: Southern Borer, Hole Borer, Malay Borer.',
  'metadata': {'category': 'durian_pest_and_disease_v1'}})

In [ ]:
# Ingest the documents
doc_ids = []
for i, sample in enumerate(samples):
    doc = db.ingest_text(sample['text'], metadata=sample['metadata'], use_colpali=False)
    doc_ids.append(doc.external_id)
    print(f"{i + 1} / {len(samples)} - Ingested document with ID: {doc.external_id}")

1 / 618 - Ingested document with ID: 5ad9fb53-4661-48d8-bdf9-d7a8418d7bd6
2 / 618 - Ingested document with ID: f5a3852e-99f8-4b37-b7b3-48aeede89c33
3 / 618 - Ingested document with ID: 953b8954-6dce-4d0d-8f86-81103869963d
4 / 618 - Ingested document with ID: dcd84325-b11f-4005-86c1-cd96d48dff6b
5 / 618 - Ingested document with ID: ab4dffa3-7dd9-4a69-a292-79072862dfec
6 / 618 - Ingested document with ID: 69056851-e141-443e-ae7a-1999018fd33d
7 / 618 - Ingested document with ID: 32724e3f-7ca7-49e4-b7b1-a317bae25d65
8 / 618 - Ingested document with ID: 10a4ad56-d0c7-4c3e-82ad-5ea4e4b8b146
9 / 618 - Ingested document with ID: 94b3ab68-7c75-47f5-a349-a9dc6ff897ff
10 / 618 - Ingested document with ID: 6cd826e4-445d-40a7-b502-22b44ca87299
11 / 618 - Ingested document with ID: 471ac0e0-ca98-4c1e-9d7d-8abf4e6d9a19
12 / 618 - Ingested document with ID: 2848dd03-16cd-42c5-acb8-ce8cc5b65c6e
13 / 618 - Ingested document with ID: 064d1966-3af1-4eba-8b3c-0a3821759396
14 / 618 - Ingested document with 

In [28]:
doc_id_to_text = {}
for i, doc_id in enumerate(doc_ids):
    doc_id_to_text[doc_id] = samples[i]

In [ ]:
print("Creating knowledge graph...")
graph = db.create_graph(name="durian_pest_and_disease_version_2", documents=doc_ids)
print(f"Created graph with name: {graph.name}")

Creating knowledge graph...
Created graph with name: durian_pest_and_disease_version_2


In [47]:
viz = db.get_graph_visualization("durian_pest_and_disease_version_2")
print(len(viz["nodes"]), "nodes", len(viz["links"]), "edges")

8806 nodes 11310 edges


In [53]:
# Example of using a graph with path information
response_with_paths = db.query(
    "What's the most popular disease on durian?",
    k=10,
    graph_name="durian_pest_and_disease_version_2",
    hop_depth=1,
    use_colpali=False,
    include_paths=True,
)

In [54]:
response_with_paths.completion

'The most popular (common and significant) disease on durian appears to be **Anthracnose**, caused by the fungus *Colletotrichum gloeosporioides*. This disease is frequently mentioned and characterized by its widespread symptoms affecting the leaves burn brown with light brown tissue and dark brown lesion borders, and it tends to spread throughout the entire tree.\n\nKey points:\n\n- Occurs in both rainy and dry seasons but is most evident during the dry season when durian is flowering and fruiting.\n- Common in the Chanee durian variety and sometimes in the Monthong variety (though less severe).\n- Spread by wind under favorable environmental conditions.\n- Prevention/control includes maintaining healthy trees (water and nutrients), and spraying preventative fungicides such as benomyl, carbendazim, or copper oxychloride during vulnerable growth stages.\n\nOther diseases like Pink Disease, Root and Foot Rot, Leaf Blight, Powdery Mildew, and Sooty Mold are also important, but Anthracnos

In [55]:
response_with_paths.sources

[ChunkSource(document_id='be97c1dc-3993-4440-88e1-dd4988b05f32', chunk_number=1, score=0.8195870589348512),
 ChunkSource(document_id='d044df3c-972b-4de6-b3e5-8695328918c8', chunk_number=1, score=0.8195870589348512),
 ChunkSource(document_id='3421014d-a9c2-48b5-b1a5-a71599eac2e4', chunk_number=1, score=0.8195870589348512),
 ChunkSource(document_id='72c7ed30-1815-400c-9f45-87435548ff2a', chunk_number=0, score=0.8076117795127336),
 ChunkSource(document_id='0c3994e7-540f-4ae8-b824-fb700427fc7b', chunk_number=0, score=0.8076117795127336),
 ChunkSource(document_id='5a97c303-27d2-4a70-916a-77bfa933436b', chunk_number=0, score=0.8076117795127336),
 ChunkSource(document_id='be97c1dc-3993-4440-88e1-dd4988b05f32', chunk_number=0, score=0.7870561781344797),
 ChunkSource(document_id='3421014d-a9c2-48b5-b1a5-a71599eac2e4', chunk_number=0, score=0.7870561781344797),
 ChunkSource(document_id='d044df3c-972b-4de6-b3e5-8695328918c8', chunk_number=0, score=0.7870561781344797),
 ChunkSource(document_id='75

In [56]:
response_with_paths.metadata['graph']['paths']

['Durian -> (grows well in, 1 shared chunks) -> humid tropical regions',
 'Durian -> (requires, 1 shared chunks) -> rainfall',
 'Durian -> (requires, 1 shared chunks) -> air humidity',
 'Durian -> (optimal growth at, 1 shared chunks) -> optimal temperature',
 'Durian -> (requires, 1 shared chunks) -> suitable soil']

In [57]:
stats = db.get_usage_stats()

In [58]:
stats

{'get_graph_visualization': 0, 'query': 21}

In [ ]:
import time

questions = [
    "In the first-day “Next schedule”, what happens between 11 : 30 – 12 : 00?",
    "List any four of VNS’s core values.",
    "How many on-top annual-leave days do employees get, anAd how many can be carried over?",
    "Quote two purposes for which VNS collects personal information.",
    "What does the Policy call the unique files (‘cookies’) stored on your device?",
    "Who's the CEO of VNS",
    "Who holds the Managing Director role at VNS?",
    "On the first-day schedule, what activity happens from 10 : 15 – 11 : 30?"
]

for i, question in enumerate(questions):
    print("------------------------------")
    print(f"Sample {i + 1} / {len(questions)}")
    print(f"Question {question}")
    try:
        t1 = time.time()
        response = db.query(
            question,
            k=20,
            graph_name="updated_vns_policy",
            hop_depth=1,
            use_colpali=False,
            include_paths=True,
        )
        t2 = time.time()
        print(f"Answer: {response.completion}")
        print(f"Response Time: {t2 - t1} seconds")
        print(f"Paths: {response.metadata['graph']['paths']}")
        print("------------------------------")
        print("\n")
    except Exception as e:
        print(
            f"Error: {e}"
        )
        print("------------------------------")
        print("\n")

------------------------------
Sample 1 / 8
Question In the first-day “Next schedule”, what happens between 11 : 30 – 12 : 00?
Answer: The provided context does not include details about the "Next schedule" or any specific events or activities planned between 11:30 – 12:00 on the first day. Therefore, I cannot determine what happens during that time slot based on the given information.
Response Time: 6.853302955627441 seconds
Paths: []
------------------------------


------------------------------
Sample 2 / 8
Question List any four of VNS’s core values.
Error: timed out
------------------------------


------------------------------
Sample 3 / 8
Question How many on-top annual-leave days do employees get, anAd how many can be carried over?
Answer: The provided context does not include information about the number of on-top annual-leave days employees get or how many they can carry over.

If you have other documents or additional context regarding employee leave policies, please provi

In [5]:
import json

json_path = "/Users/mac/Documents/PHUNGPX/knowledge_graph_searching/examples/data/QA_17_pest_disease_predict.json"
with open(json_path, "r", encoding="utf-8") as file:
    data = json.load(file)

In [6]:
data[0]

{'question': 'My young durian leaves are curling and look scorched at the edges—could that be leafhopper damage and what should I do first?',
 'answer': "Yes, leafhopper feeding injects toxins that cause 'hot-water scald' and edge burn. Start with sticky traps and close scouting, then consider biocontrols like Beauveria/Metarhizium before rotating listed insecticides if needed.",
 'citation': ['PL-DR-DP-ED-05-เพลี้ยจักจั่นฝอยทุเรียน.pdf | damage & control | symptom sequence + control options'],
 'type': 'multi-hop',
 'batch': 'a',
 'predict': 'The symptoms you describe—young durian leaves curling and having scorched edges—are consistent with damage caused by leafhoppers.\n\n**Identification & Symptoms:**\n- Leafhopper nymphs and adults feed by inserting their piercing mouthparts and sucking sap from durian leaves.\n- This causes the leaves to curl and the edges to scorch or brown.\n- Nymphs cause the most damage.\n- Sticky substances they secrete promote sooty mold growth, further harm

In [18]:
import time
from morphik import Morphik

# Connect to Morphik
db = Morphik(uri=None, timeout=100, is_local=True)


def save_json(data: dict, path: str):
    with open(path, "w") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)


for i, sample in enumerate(data[170:]):
    print("------------------------------")
    print(f"Sample {i + 1} / {len(data)}")
    question = sample['question']

    if sample['successed'] == True:
        print(f"Answer: {sample['predict']}")
        print("------------------------------")
        print("\n")
        continue

    print(f"Question: {question}")
    try:
        t1 = time.time()
        response = db.query(
            question,
            k=2,
            graph_name="durian_pest_and_disease_version_2",
            hop_depth=1,
            use_colpali=False,
            use_reranking=True,
            include_paths=True,
        )
        t2 = time.time()
        sample['predict'] = response.completion
        sample['response_time'] = t2 - t1
        sample['traserval_paths'] = response.metadata['graph']['paths']
        sample['successed'] = True
        print(f"Answer: {response.completion}")
        print(f"Response Time: {t2 - t1} seconds")
        print("------------------------------")
        print("\n")
    except Exception as e:
        sample['predict'] = f"Error: {e}"
        sample['successed'] = False
        print(f"Error: {e}")
        print("------------------------------")
        print("\n")

    save_json(data, "/Users/mac/Documents/PHUNGPX/knowledge_graph_searching/examples/data/QA_17_pest_disease_predict.json")


------------------------------
Sample 1 / 578
Question: How does weed control support disease prevention?
Error: timed out
------------------------------


------------------------------
Sample 2 / 578
Answer: The recommended soil pH for durian to reduce disease is approximately **5.5 to 6.5**.

According to the provided context, durian grows best in well-drained soil with a pH of about 5.5–6.5, which is also associated with disease prevention, particularly for root and foot rot disease caused by Phytophthora palmivora. Improving soil by adding compost and manure and adjusting the pH within this range helps reduce pathogen presence and supports plant health, thus lowering disease risk. 

This is supported by the prevention guidelines for root and foot rot disease:
> "Improve the soil by adding compost and manure, and adjust the soil condition to a pH of 5.5-6.5." 

Therefore, maintaining soil pH within 5.5 to 6.5 is recommended for durian cultivation to help reduce disease incidence.
-